In [ ]:
import os 
import getpass
import pydantic
from dotenv import load_dotenv
from crewai import LLM, Agent, Task, Crew
from chromadb.config import Settings
from crewai.rag.chromadb.config import ChromaDBConfig
from chromadb.config import Settings
from crewai.tools import tool

from com.example.agentic.agent.LLMManager import LLMManager
llm = LLMManager.create_llm('openai')
llm.call('Hello')

In [ ]:
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage

# Create custom storage with specific embedder
# custom_storage = KnowledgeStorage(
#     embedder={
#         "provider": "ollama",
#         "config": {"model": "mxbai-embed-large"}
#     },
#     collection_name="my_custom_knowledge"
# )

text_kw_source = TextFileKnowledgeSource(
    file_paths=[
        "designs.txt", 
        "machine_learning.txt",
        "python_intro.txt"
    ]
)


[2026-04-23 11:09:28][ERROR]: File not found: knowledge/file:/home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/text/designs.txt. Try adding sources to the knowledge directory. If it's inside the knowledge directory, use the relative path.


FileNotFoundError: File not found: knowledge/file:/home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/text/designs.txt

In [ ]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool
from pydantic import BaseModel, Field
from typing import List, Dict


class ScanPoint(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    summary: str = Field(description="The key findings or insights about this topic")
    benefits: List[Dict[str, str]] = Field(
        description="The key benifit and details for topic",
        default_factory=list
    )
    references: List[str] = Field(
        description="the key reference design diagram url for discussed topic",
        default_factory=list
    )

In [ ]:
from crewai_tools import FileReadTool

# Initialize the tool to read any files the agents knows or lean the path for
file_read_tool = FileReadTool()

# OR

# Initialize the tool with a specific file path, so the agent can only read the content of the specified file
file_read_tool = FileReadTool(file_path='path/to/your/file.txt')

In [ ]:
from com.example.agentic.tools.config import _embedder_config_ollama,_embedder_config_openai

# Agent city expert
scan_doc_agent = Agent(
    role="As a software architect with expertise in {topic}",
    goal="Design robust, scalable system architectures that balance performance, maintainability, and cost-effectiveness",
    backstory="With 20+ years of experience building {topic} systems at scale, you've developed a pragmatic approach"
              "to software architecture. You've seen both successful and failed systems and have learned valuable lessons from each." 
              "You balance theoretical best practices with practical constraints and always consider the maintenance and operational" 
              "aspects of your designs."
    tools=[],
    verbose=True,
    knowledge_sources=[text_kw_source],
    embedder=_embedder_config_openai,
    llm=llm,
    allow_delegation=False,
)


# Define Tasks
scan_doc_task = Task(
    description='Extract key Benefits for {topic} and provide a summarized report.',
    expected_output='Output should include topic benifit, summary. output should include references as list',
    agent=scan_doc_agent,
    output_json=ScanPoint,
    output_file="outputs/L006/scan_doc-summary_{run_id}.json"
)


# Review the context you got and expands each topic into full section for a report about {topic}
# Make sure you find top 10 interesting and relevant design url about {topic} and return list of url
# Define Tasks
format_doc_task = Task(
    description='Extract key Benefits for {topic} and provide a summarized report.',
    expected_output='Output should include topic benifit, summary. output should include references as list',
    agent=scan_doc_agent,
    output_json=ScanPoint,
    output_file="outputs/L006/format_doc-summary_{run_id}.json" 
)

In [ ]:

# Assemble the Crew
from crewai import Crew, Process
from datetime import datetime

_exe_date = datetime.now().strftime('%Y-%m-%d')
_exe_id = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Crew triggered on {_exe_date} with execution id {_exe_id}")

crew = Crew(
    agents=[scan_doc_agent],
    tasks=[scan_doc_task],
    process=Process.sequential,
    share_crew=False,
    verbose=True
)

inputs = {
    'topic': 'Microservice Design',
    'date': _exe_date,
    'run_id': _exe_id
}

# Execute the Tasks
result = crew.kickoff(inputs=inputs)
print(result)

In [1]:
from crewai_tools import RagTool
from com.example.agentic.tools.config import _tool_config,_rag_tool_config

# Create a RAG tool with custom configuration
rag_tool = RagTool(config=_rag_tool_config, summarize=True)



In [2]:
rag_tool.add(file_path='/home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/text/Brijesh_Resume.txt')

CREWAI_TOOLS_ALLOW_UNSAFE_PATHS is enabled — skipping file path validation for: /home/brijeshdhaker/IdeaProjects/crewai_design_document/docs/text/Brijesh_Resume.txt


In [4]:
r = rag_tool.run("JPMORGAN")
r

"Relevant Content:\n\n• Cloud/Data: HDFS, ADLS, AWS S3, ADX, ADF, Azure Databricks, Spark, Confluent Kafka, Flink\n\n• Microservices: SpringBoot, REST APIs, JWT, OAuth\n\n• DevOps: Docker, Kubernetes, EKS, AKS\n\nProfessional Certifications\n\n•\n\n•\n\n•\n\n•\n\n•\n\n•\n\n•\n\n•\n\n•\n\n•\n\nDatabricks Datalake Platform — Databricks, APR 2026\n\nDatabricks AI Agents Fundamentals — Databricks, APR 2026\n\nDatabricks Generative AI Fundamentals — Databricks, APR 2025\n\nAzure Cloud Fundamentals — Microsoft, APR 2025\n\nAzure Data Fundamentals — Microsoft, APR 2025\n\nAzure AI Fundamentals — Microsoft, APR 2025\n\nAWS Solution Architect Associate — AWS, Jan 2022\n\nKubernetes Application Developer — Linux Foundation, Jan 2022\n\nCertified Scrum Master — Scrum Alliance, APR 2015\n\nCertification in Financial Market — NSE, APR 2008\n\nDomain Knowledge\n\n•FINANCIAL PRODUCTS & PORTFOLIO RISK\n\n•MARKET REGULATORY REPORTING\n\n•GROUP COMPLIANCE & OPERATING RISK\n\nEducation\n\nJUN 2005\n\nBac

In [15]:
from github import Github, Auth
from langchain_community.agent_toolkits.github.toolkit import GitHubToolkit
from langchain_community.utilities.github import GitHubAPIWrapper

github = GitHubAPIWrapper()
toolkit = GitHubToolkit.from_github_api_wrapper(github)

_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GIT_HUB_TOKEN")

# Use the new Auth class to handle your token
auth = Auth.Token(_git_hub_token)

# Authenticate using a Personal Access Token
github = Github(auth=auth)

# Get a specific repository
repo = github.get_repo(_repo)

print(repo.stargazers_count)

ValidationError: 1 validation error for GitHubAPIWrapper
  Value error, Did not find github_repository, please add an environment variable `GITHUB_REPOSITORY` which contains it, or pass `github_repository` as a named parameter. [type=value_error, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error

In [16]:
from langchain_community.agent_toolkits.github.toolkit import GitHubToolkit
from langchain_community.utilities.github import GitHubAPIWrapper

_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GIT_HUB_TOKEN")

github = GitHubAPIWrapper(
    active_branch="main", 
    github_repository="brijeshdhaker/docker-hadoop-cluster",
    
)
# The wrapper automatically looks for GITHUB_PERSONAL_ACCESS_TOKEN if not passed
toolkit = GitHubToolkit.from_github_api_wrapper(github)
tools = toolkit.get_tools()
for tool in tools:
    print(tool.name)

ValidationError: 1 validation error for GitHubAPIWrapper
  Value error, Did not find github_app_id, please add an environment variable `GITHUB_APP_ID` which contains it, or pass `github_app_id` as a named parameter. [type=value_error, input_value={'active_branch': 'main',.../docker-hadoop-cluster'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error

In [8]:
from langchain_community.document_loaders import GithubFileLoader

_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")

loader = GithubFileLoader(
    repo=_repo,  # the repo name
    branch="main",  # the branch name
    access_token=_git_hub_token,
    github_api_url="https://api.github.com",
    file_filter=lambda file_path: file_path.endswith(
        ".md"
    ),  # load all markdowns files.
)
documents = loader.load()

In [18]:
documents[1].page_content

'---------------------------\nCloudera QuickStart VM 6.3.2\nCentOS 7 + GNOME Based\nJava Eclipse & Scala Eclipse IDE Included\nMySql With \'retail_db\' Installed\n\n\n---------------------------\nMinimum System Requirement - 2/4 Cores + 16GB RAM\nhttps://www.peazip.org/\nhttp://bit.ly/GetVMPlayer\n\n---------------------------\nDownload In 3 Parts (Direct Links)\nPart 1 - https://bit.ly/CDH_6_3_2_CentOS7_1\nPart 2 - https://bit.ly/CDH_6_3_2_CentOS7_2\nPart 3 - https://bit.ly/CDH_6_3_2_CentOS7_3\nOR \nhttp://bit.ly/Minus1By12Repo\n\n\n---------------------------\nCentOS GUI Login \'Base User\' Password - BaseUser@123\n\'root\'  Password - BaseUser@123\n\n#\n#---------------------------\n#\nsudo user - osboxes\nsudo password - BaseUser@123\n\n#\n#---------------------------\n#\nMySql user - root\nMySql password - bigdata\n---------------------------\nCloudera Manager user - admin\nCloudera Manager password - admin\n---------------------------\nhttp://www.Minus1By12.com  \nhttp://bit.ly/M

In [6]:
from crewai_tools import GithubSearchTool

_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")

# Initialize the tool with your PAT
github_tool = GithubSearchTool(
    config=_rag_tool_config,
    github_repo='https://github.com/brijeshdhaker/docker-hadoop-cluster',
    gh_token=_git_hub_token,
    content_types=['code', 'issue']
)

# Use the tool within an Agent
# from crewai import Agent

# github_specialist = Agent(
#     role='GitHub Researcher',
#     goal='Search for specific code patterns and issues in the repository',
#     backstory='Expert in navigating complex codebases and tracking development issues.',
#     tools=[github_tool],
#     verbose=True
# )




CREWAI_TOOLS_ALLOW_UNSAFE_PATHS is enabled — skipping URL validation for: https://github.com/https://github.com/brijeshdhaker/docker-hadoop-cluster


In [7]:
github_tool.add(repo='brijeshdhaker/docker-hadoop-cluster',content_types=['code', 'issue'])
github_tool.run('Search for specific code patterns and issues in the repository')

CREWAI_TOOLS_ALLOW_UNSAFE_PATHS is enabled — skipping URL validation for: https://github.com/brijeshdhaker/docker-hadoop-cluster


"Relevant Content:\n\n• Cloud/Data: HDFS, ADLS, AWS S3, ADX, ADF, Azure Databricks, Spark, Confluent Kafka, Flink\n\n• Microservices: SpringBoot, REST APIs, JWT, OAuth\n\n• DevOps: Docker, Kubernetes, EKS, AKS\n\nProfessional Certifications\n\n•\n\n•\n\n•\n\n•\n\n•\n\n•\n\n•\n\n•\n\n•\n\n•\n\nDatabricks Datalake Platform — Databricks, APR 2026\n\nDatabricks AI Agents Fundamentals — Databricks, APR 2026\n\nDatabricks Generative AI Fundamentals — Databricks, APR 2025\n\nAzure Cloud Fundamentals — Microsoft, APR 2025\n\nAzure Data Fundamentals — Microsoft, APR 2025\n\nAzure AI Fundamentals — Microsoft, APR 2025\n\nAWS Solution Architect Associate — AWS, Jan 2022\n\nKubernetes Application Developer — Linux Foundation, Jan 2022\n\nCertified Scrum Master — Scrum Alliance, APR 2015\n\nCertification in Financial Market — NSE, APR 2008\n\nDomain Knowledge\n\n•FINANCIAL PRODUCTS & PORTFOLIO RISK\n\n•MARKET REGULATORY REPORTING\n\n•GROUP COMPLIANCE & OPERATING RISK\n\nEducation\n\nJUN 2005\n\nBac